In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import datasets

In [2]:
dataset = datasets.load_dataset("google/code_x_glue_cc_code_refinement", "medium").shuffle()

In [3]:
def letter_tokenizer(s: str):
    return list(s) + ["EOS"]

def word_tokenizer(s: str):
    return s.split() + ["EOS"]

vocab = set()
def map_fn(x: dict[str, list[str]]):
    # x: {"buggy": [str], "fixed": [str]}
    vocab.update("".join(x["buggy"]) + "".join(x["fixed"]))
    return {
        "buggy": [letter_tokenizer(f) for f in x["buggy"]],
        "fixed": [letter_tokenizer(f) for f in x["fixed"]],
    }

dataset = dataset.map(map_fn, batched=True)
vocab.add("EOS")

print(dataset["train"][0])

Map:   0%|          | 0/52364 [00:00<?, ? examples/s]

Map:   0%|          | 0/6546 [00:00<?, ? examples/s]

Map:   0%|          | 0/6545 [00:00<?, ? examples/s]

{'id': 48771, 'buggy': ['p', 'u', 'b', 'l', 'i', 'c', ' ', 'v', 'o', 'i', 'd', ' ', 'M', 'E', 'T', 'H', 'O', 'D', '_', '1', ' ', '(', ' ', ')', ' ', '{', ' ', 'f', 'o', 'r', ' ', '(', ' ', 'i', 'n', 't', ' ', 'i', ' ', '=', ' ', '0', ' ', ';', ' ', 'i', ' ', '<', ' ', '(', ' ', 'V', 'A', 'R', '_', '1', ' ', '.', ' ', 'M', 'E', 'T', 'H', 'O', 'D', '_', '2', ' ', '(', ' ', ')', ' ', ')', ' ', ';', ' ', 'i', ' ', '+', '+', ' ', ')', ' ', '{', ' ', 'V', 'A', 'R', '_', '2', ' ', '.', ' ', 'g', 'e', 't', ' ', '(', ' ', 'i', ' ', ')', ' ', '.', ' ', 'M', 'E', 'T', 'H', 'O', 'D', '_', '1', ' ', '(', ' ', ')', ' ', ';', ' ', 'V', 'A', 'R', '_', '3', ' ', '.', ' ', 'g', 'e', 't', ' ', '(', ' ', 'i', ' ', ')', ' ', '.', ' ', 'M', 'E', 'T', 'H', 'O', 'D', '_', '1', ' ', '(', ' ', ')', ' ', ';', ' ', '}', ' ', 'V', 'A', 'R', '_', '4', ' ', '=', ' ', 'n', 'u', 'l', 'l', ' ', ';', ' ', '}', '\n', 'EOS'], 'fixed': ['p', 'u', 'b', 'l', 'i', 'c', ' ', 'v', 'o', 'i', 'd', ' ', 'M', 'E', 'T', 'H', 'O', 'D

In [4]:
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")

Vocab size: 87


In [5]:
ohe = {c: i for i, c in enumerate(vocab)}
def ohe_map_fn(x: dict[str, list[str]]):
    return {
        "buggy": [[ohe[c] for c in f]  for f in x["buggy"]],
        "fixed": [[ohe[c] for c in f]  for f in x["fixed"]],
    }

dataset = dataset.map(ohe_map_fn, batched=True, batch_size=5000, remove_columns=["id"])

Map:   0%|          | 0/52364 [00:00<?, ? examples/s]

Map:   0%|          | 0/6546 [00:00<?, ? examples/s]

Map:   0%|          | 0/6545 [00:00<?, ? examples/s]

In [8]:
print(dataset["train"][0])

{'buggy': [55, 50, 16, 69, 67, 27, 77, 32, 81, 67, 37, 77, 53, 22, 39, 3, 6, 74, 30, 61, 77, 63, 77, 43, 77, 23, 77, 66, 81, 59, 77, 63, 77, 67, 62, 71, 77, 67, 77, 84, 77, 12, 77, 72, 77, 67, 77, 1, 77, 63, 77, 85, 21, 28, 30, 61, 77, 19, 77, 53, 22, 39, 3, 6, 74, 30, 5, 77, 63, 77, 43, 77, 43, 77, 72, 77, 67, 77, 58, 58, 77, 43, 77, 23, 77, 85, 21, 28, 30, 5, 77, 19, 77, 45, 49, 71, 77, 63, 77, 67, 77, 43, 77, 19, 77, 53, 22, 39, 3, 6, 74, 30, 61, 77, 63, 77, 43, 77, 72, 77, 85, 21, 28, 30, 75, 77, 19, 77, 45, 49, 71, 77, 63, 77, 67, 77, 43, 77, 19, 77, 53, 22, 39, 3, 6, 74, 30, 61, 77, 63, 77, 43, 77, 72, 77, 18, 77, 85, 21, 28, 30, 15, 77, 84, 77, 62, 50, 69, 69, 77, 72, 77, 18, 78, 13], 'fixed': [55, 50, 16, 69, 67, 27, 77, 32, 81, 67, 37, 77, 53, 22, 39, 3, 6, 74, 30, 61, 77, 63, 77, 43, 77, 23, 77, 66, 81, 59, 77, 63, 77, 67, 62, 71, 77, 67, 77, 84, 77, 12, 77, 72, 77, 67, 77, 1, 77, 63, 77, 85, 21, 28, 30, 61, 77, 19, 77, 53, 22, 39, 3, 6, 74, 30, 5, 77, 63, 77, 43, 77, 43, 77,

In [9]:
train_dataloader, val_dataloader, test_dataloader = [
    torch.utils.data.DataLoader(
        dataset[split]
    ) for split in ["train", "validation", "test"]
]

print(f"Train: {len(train_dataloader.dataset)}")
print(train_dataloader.dataset[0])

Train: 52364
{'buggy': [55, 50, 16, 69, 67, 27, 77, 32, 81, 67, 37, 77, 53, 22, 39, 3, 6, 74, 30, 61, 77, 63, 77, 43, 77, 23, 77, 66, 81, 59, 77, 63, 77, 67, 62, 71, 77, 67, 77, 84, 77, 12, 77, 72, 77, 67, 77, 1, 77, 63, 77, 85, 21, 28, 30, 61, 77, 19, 77, 53, 22, 39, 3, 6, 74, 30, 5, 77, 63, 77, 43, 77, 43, 77, 72, 77, 67, 77, 58, 58, 77, 43, 77, 23, 77, 85, 21, 28, 30, 5, 77, 19, 77, 45, 49, 71, 77, 63, 77, 67, 77, 43, 77, 19, 77, 53, 22, 39, 3, 6, 74, 30, 61, 77, 63, 77, 43, 77, 72, 77, 85, 21, 28, 30, 75, 77, 19, 77, 45, 49, 71, 77, 63, 77, 67, 77, 43, 77, 19, 77, 53, 22, 39, 3, 6, 74, 30, 61, 77, 63, 77, 43, 77, 72, 77, 18, 77, 85, 21, 28, 30, 15, 77, 84, 77, 62, 50, 69, 69, 77, 72, 77, 18, 78, 13], 'fixed': [55, 50, 16, 69, 67, 27, 77, 32, 81, 67, 37, 77, 53, 22, 39, 3, 6, 74, 30, 61, 77, 63, 77, 43, 77, 23, 77, 66, 81, 59, 77, 63, 77, 67, 62, 71, 77, 67, 77, 84, 77, 12, 77, 72, 77, 67, 77, 1, 77, 63, 77, 85, 21, 28, 30, 61, 77, 19, 77, 53, 22, 39, 3, 6, 74, 30, 5, 77, 63, 77, 43

In [ ]:
class LSTMModel(torch.nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim)
        self.encoder_lstm = torch.nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.encoder_lstm(embedded)
        output = self.fc(lstm_out)
        return output

model = LSTMModel(vocab_size, embedding_dim=256, hidden_dim=512)